# Fine-Tuning a Language Model with LoRA and DPO

A step-by-step guide to adapting a pretrained language model to follow instructions, using techniques that run on a MacBook.

| Step | What we do | Key concept |
|------|-----------|-------------|
| 0 | Load TinyLlama | See what a pretrained model can do out of the box |
| 1 | Understand LoRA | How to train <1% of a model's parameters and still change its behavior |
| 2 | Supervised fine-tuning | Teach the model to follow instructions using LoRA |
| 3 | DPO | Teach the model to prefer good answers over bad ones |

**Requirements:** MacBook with 8GB+ RAM. All training runs on CPU or MPS (Apple Silicon GPU). No CUDA/NVIDIA needed.

## Setup

We need several libraries:
- `transformers` — load and run language models from Hugging Face
- `peft` — implements LoRA (Parameter-Efficient Fine-Tuning)
- `trl` — training helpers for SFT and DPO
- `datasets` — load training datasets from Hugging Face

In [1]:
import os
# Allow MPS to fall back to CPU for unsupported operations
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, DPOConfig, DPOTrainer

# Use MPS (Apple Silicon GPU) if available, otherwise CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')

Using device: mps


## Step 0: Load the Base Model

**What we're doing:** Loading TinyLlama (1.1B parameters) — a small pretrained language model that can run on a MacBook.

**How it works:** TinyLlama was pretrained on 3 trillion tokens of text from the internet. It learned to predict the next token given previous tokens — the same objective as our GPT from the transformer notebook, but trained on vastly more data with a much larger model. It already knows English grammar, facts, and patterns — but it hasn't been taught to follow instructions or have conversations.

**Why TinyLlama?** At 1.1B parameters, it's small enough to finetune on a laptop (~2-4GB memory with LoRA) but large enough to produce coherent text. Larger models (7B+) would need a GPU with more VRAM.

**What's a tokenizer?** The tokenizer converts text to token IDs and back. Unlike our character-level tokenizer from the transformer notebook, this uses BPE (Byte-Pair Encoding) with a ~32k vocabulary — common words like `"the"` are single tokens, while rare words are split into subwords.

In [2]:
model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# Load the tokenizer — converts text ↔ token IDs
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model in fp32 (full precision) — safest for MPS/CPU training
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,  # fp16/bf16 can cause NaN on MPS
)

# Align pad token across tokenizer, model config, and generation config
# so no component sees a mismatch
tokenizer.pad_token = tokenizer.eos_token
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
# Clear default max_length — we always set max_new_tokens explicitly
base_model.generation_config.max_length = None

total_params = sum(p.numel() for p in base_model.parameters())
print(f'Model: {model_name}')
print(f'Parameters: {total_params:,}')
print(f'Size in memory: ~{total_params * 4 / 1e9:.1f} GB (fp32)')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Parameters: 1,100,048,384
Size in memory: ~4.4 GB (fp32)


### Generate with the base model

Let's ask the base model some medical questions *before* any finetuning. TinyLlama-Chat can follow instructions but has no specialized medical training — it will give generic, often vague answers. After SFT on medical flashcards, the answers should become noticeably more specific and clinical.

In [3]:
def generate(model, prompt, max_new_tokens=100):
    """Generate text from a prompt using the model."""
    was_training = model.training
    model.eval()  # disable dropout and gradient checkpointing for generation
    
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,  # penalize repeating the same token
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    if was_training:
        model.train()  # restore training mode if it was on before
    return response.strip()

# Medical questions — compare these answers before and after LoRA finetuning
medical_prompts = [
    'What are the common symptoms of pneumonia?',
    'What is the difference between type 1 and type 2 diabetes?',
    'What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?'
]

print('--- Base model (before finetuning) ---')
for p in medical_prompts:
    print(f'\nQ: {p}')
    print(f'A: {generate(base_model, p)}')

--- Base model (before finetuning) ---

Q: What are the common symptoms of pneumonia?
A: Yes, there are several common symptoms of pneumonia:

1. Coughing up or clogging your bronchi (the tubes that carry air to and from the lungs) with mucus or phlegm. This is usually accompanied by a feeling of difficulty breathing or shortness of breath.
2. Fever (temperature higher than 38°C).
3. Chest pain, pressure, or fullness when you breathe

Q: What is the difference between type 1 and type 2 diabetes?
A: Type 1 diabetes (T1D) and type 2 diabetes (T2D) are two different types of diabetes. Here's a brief explanation:

1. Type 1 Diabetes: This form of diabetes occurs when the body stops producing insulin, which is a hormone that helps to regulate blood sugar levels in the body. In T1D, an individual develops autoimmune destruction of the pan

Q: What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?
A: Very low Mg2+ levels (Mg2+ concentration below 3.5 mEq/L) are as

## Step 1: Understanding LoRA

**What changes:** Instead of updating all 1.1 billion parameters during training, we freeze the entire model and inject tiny trainable matrices into specific layers. Only these new matrices (~0.5% of total parameters) get updated.

**The problem LoRA solves:** Fine-tuning means updating a model's weights to change its behavior. But TinyLlama has 1.1B parameters — storing a full copy of updated weights for every task would be expensive (4.4 GB per task in fp32). And updating all parameters risks "catastrophic forgetting" — the model loses its general knowledge while learning the new task.

**How LoRA works:** Every transformer layer has large weight matrices (e.g., the Q, K, V projections from our transformer notebook). A weight matrix `W` might be `2048 × 2048` = 4 million parameters. LoRA's insight: the *change* to `W` during finetuning is usually low-rank — it can be approximated by two much smaller matrices.

Instead of updating `W` directly, LoRA:
1. Freezes the original `W` (no gradients)
2. Adds two small matrices: `A` of shape `(2048, r)` and `B` of shape `(r, 2048)`, where `r` is tiny (e.g., 8)
3. The output becomes `W @ x + (B @ A) @ x`

The product `B @ A` has shape `(2048, 2048)` — same as `W` — but is defined by only `2 × 2048 × 8 = 32,768` parameters instead of 4 million. That's a **125× reduction** per layer.

**Why `r = 8` works:** The rank `r` controls how expressive the adaptation is. Research shows that finetuning changes to weight matrices tend to be low-rank — most of the change lies in a small subspace. `r = 8` is enough for most tasks. Higher `r` = more capacity but more memory and slower training.

**What is `lora_alpha`?** A scaling factor that controls how much the LoRA matrices influence the output. The actual scaling is `lora_alpha / r`. With `alpha=16, r=8`, the scaling is `2.0` — the LoRA contribution is amplified 2×. This is a tuning knob: higher alpha = LoRA has more influence on the output.

**Which layers get LoRA?** We apply it to the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) — the same Q, K, V, and output matrices from our transformer notebook. These are where the model decides what to attend to, so modifying them has the most impact on behavior.

**After training:** The LoRA adapter is a tiny file (~2-10 MB) that can be loaded on top of the frozen base model. You can have multiple adapters (one per task) sharing the same base model.

In [4]:
# Configure LoRA: which layers to adapt, and how much capacity to give them
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,       # we're doing next-token prediction
    r=8,                                 # rank of the A and B matrices — lower = fewer params
    lora_alpha=16,                       # scaling factor — actual scale is alpha/r = 2.0
    lora_dropout=0.05,                   # dropout on LoRA layers to prevent overfitting
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # attention layers only
)

# Wrap the base model with LoRA — freezes base weights, adds trainable A/B matrices
model = get_peft_model(base_model, lora_config)

# Show what happened
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


Notice the output: out of ~1.1B total parameters, only a few million are trainable. The rest are frozen — they contribute to the forward pass but don't receive gradient updates during training.

Let's inspect what LoRA actually added to the model:

In [5]:
# Look at one attention layer to see the LoRA matrices
layer = model.base_model.model.model.layers[0].self_attn

print('Q projection structure (original weight + LoRA A and B):')
print(layer.q_proj)
print()

# The LoRA matrices are tiny compared to the original weight
W = layer.q_proj.base_layer.weight
A = layer.q_proj.lora_A['default'].weight
B = layer.q_proj.lora_B['default'].weight
print(f'Original W shape: {W.shape}  ({W.numel():,} params) — FROZEN')
print(f'LoRA A shape:     {A.shape}  ({A.numel():,} params) — trainable')
print(f'LoRA B shape:     {B.shape}  ({B.numel():,} params) — trainable')
print(f'LoRA adds {A.numel() + B.numel():,} params vs {W.numel():,} original — {(A.numel() + B.numel()) / W.numel() * 100:.1f}% of the layer')

Q projection structure (original weight + LoRA A and B):
lora.Linear(
  (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.05, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=2048, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=2048, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)

Original W shape: torch.Size([2048, 2048])  (4,194,304 params) — FROZEN
LoRA A shape:     torch.Size([8, 2048])  (16,384 params) — trainable
LoRA B shape:     torch.Size([2048, 8])  (16,384 params) — trainable
LoRA adds 32,768 params vs 4,194,304 original — 0.8% of the layer


## Step 2: Supervised Fine-Tuning (SFT)

**What changes:** We train the LoRA adapter on a dataset of medical Q&A pairs so the model learns to answer medical questions. This is a good test for LoRA because the base model has general knowledge but no specialized medical training — any medical expertise in the output comes from the LoRA adapter.

**How it works:** SFT is the same next-token prediction training from our transformer notebook — the model sees a medical question + answer, and learns to predict each token of the answer given everything before it. The only difference is that we're updating the LoRA matrices (A and B) instead of all the weights.

**The training data:** We use medical flashcards — short question/answer pairs like "What are the symptoms of pneumonia?" → "Cough, fever, shortness of breath...". We format them as chat messages so they match the template the model was trained on.

**What backprop looks like with LoRA:** During the forward pass, each attention layer computes `W @ x + (B @ A) @ x`. During backprop, gradients flow back through `B @ A` and update only `A` and `B` — the original `W` is frozen (its `requires_grad` is `False`). This is why LoRA is fast: we only compute and store gradients for ~0.5% of the parameters.

**Why we use SFTTrainer:** We could write a training loop by hand (like in the transformer notebook), but `SFTTrainer` from the `trl` library handles chat template formatting, tokenization, padding, and batching. It's the standard tool for instruction-tuning.

In [6]:
# Load medical flashcards — short Q&A pairs about medical topics
dataset = load_dataset('medalpaca/medical_meadow_medical_flashcards', split='train')

# Use a small subset — enough to see the effect, fast enough for a laptop
dataset = dataset.select(range(500))

# Format as chat messages so SFTTrainer can apply the model's chat template
def format_medical(example):
    return {
        'messages': [
            {'role': 'user', 'content': example['input']},
            {'role': 'assistant', 'content': example['output']},
        ]
    }

dataset = dataset.map(format_medical, remove_columns=dataset.column_names)

print(f'Training examples: {len(dataset)}')
print(f'\nExample:')
for msg in dataset[0]['messages']:
    text = msg['content'][:120] + '...' if len(msg['content']) > 120 else msg['content']
    print(f"  [{msg['role']}]: {text}")

Training examples: 500

Example:
  [user]: What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?
  [assistant]: Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.


In [7]:
# Configure training — tuned for 16GB MacBook
sft_config = SFTConfig(
    output_dir='./sft-output',
    num_train_epochs=1,              # one pass through the data
    per_device_train_batch_size=1,   # minimal batch size to save memory
    gradient_accumulation_steps=8,   # simulate batch_size=8 over multiple steps
    learning_rate=2e-4,              # higher than pretraining — LoRA can handle it
    max_length=256,                  # shorter sequences use less memory
    logging_steps=10,                # log loss every 10 steps so we can track progress
    save_strategy='no',              # don't save checkpoints (saves disk space)
    fp16=False, bf16=False,          # fp32 only — fp16/bf16 cause NaN on MPS
    gradient_checkpointing=True,     # trade compute for memory — recompute activations instead of storing them
    dataloader_pin_memory=False,     # pin_memory not supported on MPS
    report_to='none',               # disable wandb/mlflow logging
    disable_tqdm=False,              # show progress bar
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print(f'Training {trainer.args.num_train_epochs} epoch(s), '
      f'{len(trainer.get_train_dataloader())} steps, '
      f'logging every {sft_config.logging_steps} steps')
trainer.train()
print('Done!')

Training 1 epoch(s), 500 steps, logging every 10 steps


Step,Training Loss
10,1.361991
20,1.049695
30,1.039849
40,0.991698
50,1.006071
60,0.979836


Done!


### Generate after SFT

The LoRA adapter has been trained on 500 medical flashcards. The base weights are unchanged — only the small A and B matrices were updated. Let's ask the same medical questions and compare.

We also save the adapter so the DPO step can continue from it.

In [8]:
from IPython.display import display, Markdown

# Save the SFT adapter — DPO will load this to continue from where SFT left off
model.save_pretrained('./sft-adapter')
print(f'SFT adapter saved to ./sft-adapter/')

# Generate answers from both models and display side by side
rows = ['| Question | Base Model | After SFT |\n|----------|------------|-----------|']
for p in medical_prompts:
    base_answer = generate(base_model, p).replace('\n', ' ').replace('|', '\\|')
    sft_answer = generate(model, p).replace('\n', ' ').replace('|', '\\|')
    rows.append(f'| {p} | {base_answer[:200]} | {sft_answer[:200]} |')

display(Markdown('\n'.join(rows)))

SFT adapter saved to ./sft-adapter/


| Question | Base Model | After SFT |
|----------|------------|-----------|
| What are the common symptoms of pneumonia? | Pneumonia is a lung infection caused by bacteria or fungi, and it can be diagnosed based on several specific symptoms. These include fever, cough, chest pain, shortness of breath, wheezing, and decrea | Pneumonia is a disease that affects the lungs and can result in several complications, including inflammation and swelling. Some common symptoms of pneumonia include coughing up blood or mucus, chest  |
| What is the difference between type 1 and type 2 diabetes? | Type 1 diabetes is a autoimmune disease that occurs when the pancreas produces little or no insulin, while type 2 diabetes is caused by the body's inability to produce enough insulin. | Type 1 diabetes occurs when the immune system attacks and destroys insulin-producing cells in the pancreas, leading to a lack of insulin production. Type 2 diabetes occurs when the body becomes resist |
| What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels? | Very low Mg2+ levels are associated with high PTH levels. This can lead to hyperparathyroidism. Low levels of calcium in the bloodstream can also result from this condition. The exact mechanisms by wh | Very low Mg2+ levels can cause a rise in plasma calcium levels as well as increased activity of enzymes involved in bone remodeling. This increase in bone resorption may result in decreased bone miner |

## Step 3: DPO (Direct Preference Optimization)

**What changes:** Instead of showing the model "here's the correct answer" (SFT), we show it pairs of answers and say "this one is better than that one." The model learns to prefer good responses over bad ones.

**How this builds on SFT:** DPO continues from the SFT-trained adapter, not from scratch. We saved the SFT adapter in the previous step, and now load it so DPO refines those medical Q&A weights further. The full pipeline is: base model → SFT (learn to answer) → DPO (learn to answer *well*).

**The problem DPO solves:** SFT teaches the model to imitate the training data — but not all responses are equally good. Given "Explain gravity", one response might be clear and accurate while another is vague and rambling. SFT treats both the same if they appear in the training data. DPO explicitly teaches the model to distinguish good from bad.

**How DPO works:** The training data contains triplets: (prompt, chosen response, rejected response). For each example, DPO:
1. Runs the prompt through the current model and a frozen reference model (a copy before DPO training)
2. Computes the probability both models assign to the chosen vs. rejected response
3. Updates the LoRA weights so the current model assigns higher probability to the chosen response, relative to the reference model

The reference model acts as an anchor — it prevents the model from drifting too far from its pretrained knowledge. Without it, the model could learn to always output "Yes!" if that happened to be the chosen response, forgetting everything else.

**Memory trick with LoRA:** Normally DPO would load two full copies of the model (~8.8GB). But when using LoRA, `DPOTrainer` can use the frozen base weights as the reference — no second copy needed. We pass `ref_model=None` to enable this. The reference is just the base model without the LoRA adapter applied, which is already in memory.

**Why DPO instead of RLHF?** Traditional RLHF (Reinforcement Learning from Human Feedback) requires training a separate reward model, then using PPO (a reinforcement learning algorithm) to optimize against it — complex and unstable. DPO achieves similar results with a single training loop and no reward model. It reformulates the RL objective directly into a classification loss on preference pairs.

**The `beta` parameter:** Controls how much the model can diverge from the reference. Higher beta = stay closer to the reference (more conservative). Lower beta = allow more change (riskier but potentially better). `beta=0.1` is the standard default.

**Note:** We free the SFT model from memory before loading the DPO model to avoid running out of RAM.

In [9]:
# Load a preference dataset: each example has a prompt, a "chosen" (good) response,
# and a "rejected" (bad) response
dpo_dataset = load_dataset('argilla/distilabel-intel-orca-dpo-pairs', split='train')

# Filter to only examples with clear quality differences
dpo_dataset = dpo_dataset.filter(lambda x: x['status'] != 'tie')

# Use a small subset for laptop training
dpo_dataset = dpo_dataset.select(range(200))

# Format as chat messages — DPOTrainer expects 'prompt', 'chosen', 'rejected' as message lists.
# Using the chat format avoids tokenization mismatches between prompt and prompt+response.
def format_dpo(example):
    return {
        'prompt': [{'role': 'user', 'content': example['input']}],
        'chosen': [{'role': 'assistant', 'content': example['chosen']}],
        'rejected': [{'role': 'assistant', 'content': example['rejected']}],
    }

dpo_dataset = dpo_dataset.map(format_dpo, remove_columns=dpo_dataset.column_names)

print(f'DPO training examples: {len(dpo_dataset)}')
print(f'\nExample:')
print(f"  Prompt:   {dpo_dataset[0]['prompt'][0]['content'][:100]}...")
print(f"  Chosen:   {dpo_dataset[0]['chosen'][0]['content'][:100]}...")
print(f"  Rejected: {dpo_dataset[0]['rejected'][0]['content'][:100]}...")

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

DPO training examples: 200

Example:
  Prompt:   Generate an approximately fifteen-word sentence that describes all this data: Midsummer House eatTyp...
  Chosen:   Midsummer House is a moderately priced Chinese restaurant with a 3/5 customer rating, located near A...
  Rejected:  Sure! Here's a sentence that describes all the data you provided:

"Midsummer House is a moderately...


In [10]:
# Free SFT model and base model from memory before loading DPO model
import gc
from peft import PeftModel

del model, base_model, trainer
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

# Load a fresh base model, then load the SFT adapter on top
# This way DPO continues from where SFT left off (base → SFT → DPO)
dpo_base = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,
)
dpo_base.config.pad_token_id = tokenizer.pad_token_id
dpo_base.generation_config.pad_token_id = tokenizer.pad_token_id
dpo_base.generation_config.max_length = None

# Load the SFT adapter we saved earlier — the LoRA A/B matrices trained on medical data
dpo_model = PeftModel.from_pretrained(dpo_base, './sft-adapter', is_trainable=True)
print(f'Loaded SFT adapter — DPO will refine these weights')
dpo_model.print_trainable_parameters()

# DPO uses more memory than SFT (2 forward passes per example) — keep everything minimal
dpo_config = DPOConfig(
    output_dir='./dpo-output',
    num_train_epochs=1,
    per_device_train_batch_size=1,   # DPO needs more memory (runs 2 forward passes per example)
    gradient_accumulation_steps=8,   # simulate batch_size=8
    learning_rate=5e-5,              # lower LR for DPO — it's a more delicate optimization
    max_length=256,                  # shorter sequences to fit in memory
    beta=0.1,                        # KL penalty — how much the model can diverge from reference
    logging_steps=5,                 # log frequently — DPO has fewer steps
    save_strategy='no',
    fp16=False, bf16=False,
    gradient_checkpointing=True,     # recompute activations to save memory
    dataloader_pin_memory=False,     # pin_memory not supported on MPS
    report_to='none',
    remove_unused_columns=False,
    disable_tqdm=False,              # show progress bar
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None,                  # with LoRA, use frozen base weights as reference — no second model copy
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

print(f'Training {dpo_config.num_train_epochs} epoch(s), '
      f'{len(dpo_trainer.get_train_dataloader())} steps, '
      f'logging every {dpo_config.logging_steps} steps')
print('(DPO is slower — 2 forward passes per example)')
dpo_trainer.train()
print('Done!')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded SFT adapter — DPO will refine these weights
trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Training 1 epoch(s), 200 steps, logging every 5 steps
(DPO is slower — 2 forward passes per example)


Step,Training Loss
5,0.665909
10,0.612366
15,0.572651
20,0.522117
25,0.484643


Done!


### Generate after DPO

The DPO-trained model has learned to prefer higher-quality responses. We compare all three stages: base model (no adapter), after SFT (loaded from saved adapter), and after DPO. PEFT lets us load multiple adapters on the same base model and switch between them.

In [12]:
# Load the SFT adapter as a second adapter named "sft" — no extra base model copy needed
dpo_model.load_adapter('./sft-adapter', adapter_name='sft')

rows = ['| Question | Base Model | After SFT | After DPO |\n|----------|------------|-----------|-----------|']
for p in medical_prompts:
    # Base model: disable all adapters
    dpo_model.disable_adapter_layers()
    base_answer = generate(dpo_model, p).replace('\n', ' ').replace('|', '\\|')
    dpo_model.enable_adapter_layers()

    # SFT model: switch to the saved SFT adapter
    dpo_model.set_adapter('sft')
    sft_answer = generate(dpo_model, p).replace('\n', ' ').replace('|', '\\|')

    # DPO model: switch back to the default (DPO-trained) adapter
    dpo_model.set_adapter('default')
    dpo_answer = generate(dpo_model, p).replace('\n', ' ').replace('|', '\\|')

    rows.append(f'| {p} | {base_answer[:200]} | {sft_answer[:200]} | {dpo_answer[:200]} |')

display(Markdown('\n'.join(rows)))

| Question | Base Model | After SFT | After DPO |
|----------|------------|-----------|-----------|
| What are the common symptoms of pneumonia? | The following are some common symptoms of pneumonia: 1. Coughing up phlegm (flu-like cough) or blood-tinged sputum 2. Fever, chills, and shaking with chest pain 3. Difficulty breathing or wheezing in  | Pneumonia is a medical condition that causes inflammation and infection within the lungs. Common symptoms include fever, chest pain or discomfort, coughing up phlegm (mucus), shortness of breath, chil | Pneumonia is characterized by fever, chest pain or pressure, coughing up mucus, and difficulty breathing. These symptoms may occur together with other respiratory complications such as coryza (a sore  |
| What is the difference between type 1 and type 2 diabetes? | Type 1 diabetes (T1D) and type 2 diabetes are two types of diabetes that occur when the body's cells cannot produce insulin properly. In T1D, the immune system attacks the pancreas, resulting in the l | Type 1 diabetes involves a deficiency of insulin, while type 2 diabetes results from an inability to produce or use insulin. Both types can be challenging for individuals with blood sugar control issu | Type 1 diabetes, also known as insulin-dependent diabetes mellitus, is caused by an autoimmune reaction against a specific protein in the pancreas. Type 2 diabetes, on the other hand, is associated wi |
| What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels? | Very low Mg2+ levels (Mg2/Mg+) can lead to a number of negative effects on the body, including: 1. Hypocalcemia: Hypocalcemia occurs when there is too little calcium in the bloodstream due to decrease | Very low Mg2+ levels are associated with high levels of PTH (prolactin) and low calcium levels. This can lead to osteoporosis, as bone loss occurs due to a lack of calcium in the body. The combination | Very low Mg2+ levels are associated with a deficiency in calcium ion concentration, which can lead to hypercalcemia. This condition can be caused by various factors such as renal disease or other unde |

## Summary

| Step | Technique | What the model learns | Training data |
|------|-----------|----------------------|---------------|
| 0 | Base model | Already pretrained on internet text | — |
| 1 | LoRA setup | Nothing yet — we just added trainable adapters | — |
| 2 | SFT | Imitate the style and format of instruction-response pairs | (instruction, response) |
| 3 | DPO | Prefer good responses over bad ones | (prompt, chosen, rejected) |

The full pipeline in production is: **pretrain → SFT → DPO** (or RLHF). We skipped pretraining (used TinyLlama), did SFT to teach instruction-following, and DPO to improve response quality.

## Next steps

1. **Merge the adapter** — use `model.merge_and_unload()` to bake the LoRA weights into the base model for faster inference (no adapter overhead)
2. **Try different datasets** — finetune on code, medical text, or a specific domain
3. **Quantization** — use 4-bit quantization (QLoRA) to run larger models on the same hardware
4. **Evaluate properly** — use benchmarks or human evaluation instead of eyeballing outputs